In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os
from torch.utils.data import DataLoader
import seaborn as sns
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import random
import csv
import torch
import torch.nn.functional as F
import numpy as np
import os
import librosa
from torch.utils.data import Dataset
import json

In [5]:
class CNNChromagram(nn.Module):
    def __init__(self, input_time = 861):
        super(CNNChromagram, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, 3), padding = 'same', dilation = (1, 2)) # 1 x T x 12
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 3), padding = 'same', dilation  = (1, 2)) # 32 x T x 12
        self.conv3 = nn.Conv2d(64, 128, kernel_size = (3, 3), padding = 'same', dilation = (1, 2)) # 64 x T x 12

        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=(2, 1))

        fc_input_size = 64 * (25 // 4)

        self.fc1 = nn.Linear(fc_input_size, 64)

        self.fc2 = nn.Linear(64, 4)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)

        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)

        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)

        batch_size, channels, freq_bins, time_steps = x.shape

        x = x.permute(0, 3, 1, 2)
        x = x.reshape(batch_size, time_steps, -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [ ]:
class ChromaDataset(Dataset):
    def __init__(self, audio_dir, label_dir, window_size=20, sr=16000, hop_length=512, num_classes = 4, target_bins=25):
        self.audio_dir = audio_dir
        self.label_dir = label_dir
        self.window_frames = window_size * sr // hop_length
        self.sr = sr
        self.num_classes = num_classes
        self.target_bins = target_bins
        self.audio_files = [f for f in os.listdir(audio_dir)]
        with open(label_dir, "r", encoding = "utf-8") as file:
            self.label_data = json.load(file)
        self.mode_mapping = {"Unkown": 0, "우조": 1, "계면조": 2, "아니리": 3}
        self.audio_segments = self.get_segments()

    def __len__(self):
        return len(self.audio_segments)

    def get_segments(self):
        segments = []
        for file_name in self.audio_files:
            file_path = os.path.join(self.audio_dir, file_name)
            duration = self.get_audio_duration(file_path)
            num_segments = int(duration // 20)
            start_times = np.random.uniform(0, max(1, duration - 20), num_segments)
            file_segments = [(file_name, start) for start in start_times]
            segments.extend(file_segments)
        return segments

    def extract_chroma(self, file_path):
        y, _ = librosa.load(file_path, sr=self.sr)
        chroma = librosa.feature.chroma_stft(y=y, sr=self.sr)
        return chroma

    def expand_chroma(self, chroma):
        repeat = self.target_bins // chroma.shape[0] + 1
        expanded_chroma = np.tile(chroma, (repeat, 1))
        return expanded_chroma[:self.target_bins, :]

    def get_audio_duration(self, file_path):
        y, sr = librosa.load(file_path, sr = self.sr)
        return librosa.get_duration(y = y, sr = sr)

    def get_label(self, file_name, num_frames):
        hash_value = file_name.split("-")[0]
        labels = np.zeros((num_frames, self.num_classes))

        for item in self.label_data:
            if item.get("file_upload", "").startswith(hash_value):

                duration = self.get_audio_duration(os.path.join(self.audio_dir, file_name))

                for annotation in item.get("annotations", []):

                    for result in annotation.get("result", []):

                        value = result.get("value", {})
                        start_time, end_time = value.get("start", 0), value.get("end", 0)
                        mode_label = self.mode_mapping.get(value.get("labels", ["Unkwon"])[0], 0)

                        start_frame = int((start_time / duration) * num_frames)

                        end_frame = int((end_time / duration) * num_frames)

                        labels[start_frame : end_frame, mode_label] = 1

        return labels

    def __getitem__(self, idx):
        file_name, start_time = self.audio_segments[idx]
        file_path = os.path.join(self.audio_dir, file_name)
        chroma = self.extract_chroma(file_path)
        chroma = self.expand_chroma(chroma)
        num_frames = chroma.shape[1]

        labels = self.get_label(file_name, num_frames)

        if num_frames < self.window_frames:
            chroma_pad = np.zeros((25, self.window_frames - num_frames))
            chroma = np.concatenate((chroma, chroma_pad), axis=1)
            label_pad = np.zeros((self.window_frames - num_frames, self.num_classes))
            labels = np.concatenate((labels, label_pad), axis=0)

        else:
            start_idx = np.random.randint(0, num_frames - self.window_frames)
            chroma = chroma[:, start_idx:start_idx + self.window_frames]
            labels = labels[start_idx:start_idx + self.window_frames]

        time = chroma.shape[1]

        labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(0)
        #labels = F.interpolate(labels.permute(0, 2, 1), size=time, mode='linear').permute(0, 2, 1).squeeze(0)

        chroma_tensor = torch.tensor(chroma, dtype=torch.float32).unsqueeze(0)
        label_tensor = torch.tensor(labels, dtype=torch.float32)

        return chroma_tensor, label_tensor

In [36]:
audio_dir = '/home/sangheon/Desktop/Pansori/Audio'
label_dir = '/home/sangheon/Desktop/Pansori_2025_ISMIR/PansoriData/label.json'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset = ChromaDataset(audio_dir, label_dir)
song_files = dataset.audio_files
loo = LeaveOneOut()
model = CNNChromagram().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()

loo_losses = []
loo_accs = []
frame_accs = []

for train_idx, test_idx in loo.split(song_files):
    train_X, train_y = [], []
    for idx in train_idx:
        file_name = dataset.audio_files[idx]
        print(f'file_name: {file_name}')
        for segment in dataset.audio_segments:
            if segment[0] == file_name:
                segment_idx = dataset.audio_segments.index(segment)
                print(segment_idx)
                chroma, labels = dataset[segment_idx]
                train_X.append(chroma)
                train_y.append(labels)

    val_file = dataset.audio_files[test_idx[0]]

    print(f'val_file :{val_file}')

    val_X, val_y = [], []

    for segment in dataset.audio_segments:
        if segment[0] == val_file:
            segment_idx = dataset.audio_segments.index(segment)
            chroma, labels = dataset[segment_idx]
            print(segment_idx)
            val_X.append(chroma)
            val_y.append(labels)

    train_X = torch.stack(train_X).to(device)
    train_y = torch.stack(train_y).to(device)

    val_X = torch.stack(val_X).to(device)
    val_y = torch.stack(val_y).to(device)

    for epoch in range(50):
        optimizer.zero_grad()
        outputs = model(train_X)
        loss = criterion(outputs.permute(0, 2, 1), train_y.squeeze(1).argmax(dim=-1))
        loss.backward()
        optimizer.step()
    loo_losses.append(loss.item())

    model.eval()

    with torch.no_grad():
        val_output = model(val_X)
        val_loss = criterion(val_output.permute(0,2,1), val_y.squeeze(1).argmax(dim=-1))
        pred_probs = torch.softmax(val_output, dim = -1)
        pred_labels = torch.argmax(pred_probs, dim = -1)
        frame_accuracy = (pred_labels == val_y.squeeze(1).argmax(dim=-1)).float().mean(dim = 1).cpu().numpy()
        frame_accs.append(frame_accuracy)
        accuracy = frame_accuracy.mean().item()
    loo_losses.append(val_loss.item())
    loo_accs.append(accuracy)

print("\n LOO-CV Results")
print(f"Average Validation Loss: {np.mean(loo_losses):.4f}")
print(f"Average Validation Accuracy: {np.mean(loo_accs):.4f}")

Processing file: c9f91d52-03-김수연-심청가_범피중류.wav
Segments count: 27
Processing file: fe9992ba-01-김수연-심청가_청이_밥_빌러_가는데.wav
Segments count: 56
Processing file: f3e5460b-06-안향련-심청가_심청_황후가_되는_대목_엇몰이위의도_장할시고아니리이때_남경장사중머리넋이야_넋이로다.wav
Segments count: 74
Processing file: fb34b83f-08-성우향-춘향가_이도령이_천자뒤풀이를_하는_데.wav
Segments count: 98
Processing file: b35c45db-06-박초월-수궁가_고고천변.wav
Segments count: 116
Processing file: a580300d-02-정권진-심청가_뺑파_심봉사의_재산을_탕진함방아타령.wav
Segments count: 205
Processing file: ff2f07d4-04-성우향-춘향가_박석티.wav
Segments count: 219
Processing file: a35f585a-03-박초월-수궁가_토끼_용왕_속이는_대목.wav
Segments count: 248
Processing file: 30c011be-01-박초월-수궁가_별주부_토끼_유인하는_대목.wav
Segments count: 278
file_name: fe9992ba-01-김수연-심청가_청이_밥_빌러_가는데.wav
27


/tmp/ipykernel_519248/1570260776.py:97: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  label_tensor = torch.tensor(labels, dtype=torch.float32)


28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
file_name: f3e5460b-06-안향련-심청가_심청_황후가_되는_대목_엇몰이위의도_장할시고아니리이때_남경장사중머리넋이야_넋이로다.wav
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
file_name: fb34b83f-08-성우향-춘향가_이도령이_천자뒤풀이를_하는_데.wav
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
file_name: b35c45db-06-박초월-수궁가_고고천변.wav
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
file_name: a580300d-02-정권진-심청가_뺑파_심봉사의_재산을_탕진함방아타령.wav
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
file_name: ff2f07d4-04-성우향-춘향가_박석티.wav
205
206
207
208
209
210
211
212
213
214
215
216
217
218
file_name: a35f585a-03-박초월-수궁가_토끼_용왕_속이는_